# Reorder Analysis by Department ETL

## Purpose
Provide real-time reorder behavior analysis by department to identify customer loyalty patterns and high-retention categories.

## Input
* **Source:** `big_data.silver.order_products`
* **Source:** `big_data.silver.products_enriched` 

## Output
* **Target:** `big_data.gold.vw_reorder_analysis_by_department`
* **Refresh:** Real-time (always reflects current Silver data)

## SQL Logic
1. JOIN order_products with products_enriched to get department
2. GROUP BY department
3. COUNT total orders and reordered count
4. Calculate reorder_rate percentage
5. ORDER BY reorder_rate DESC

In [0]:
%sql
-- Reorder Analysis by Department View
-- Purpose: Analyze reorder behavior and customer loyalty by department

CREATE OR REPLACE VIEW big_data.gold.vw_reorder_analysis_by_department AS
SELECT 
  p.department,
  COUNT(*) AS total_orders,
  SUM(CASE WHEN op.reordered THEN 1 ELSE 0 END) AS reordered_count,
  ROUND(SUM(CASE WHEN op.reordered THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS reorder_rate
FROM big_data.silver.order_products op
JOIN big_data.silver.products_enriched p ON op.product_id = p.product_id
GROUP BY p.department
ORDER BY reorder_rate DESC;

In [0]:
%sql
-- Verify view exists and preview top 5 departments by loyalty
-- Returns 21 rows (one per department)

SELECT * FROM big_data.gold.vw_reorder_analysis_by_department
LIMIT 5;